# Procrustes Alignment Explorer

Visualises the joint sample-correspondence + orthogonal Procrustes alignment
for a single teacher–student pair from the heatmap.

Key idea: rather than averaging over 512 samples, we show the **individual matched
pairs** that the Hungarian algorithm selected — both in trajectory space (PCA
projection) and as a cost-matrix heatmap.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Matches MODEL_A / MODEL_B from compare_two_models_prep_trajectories.ipynb:
#   Model A = teacher ablated on direction 0
#   Model B = student_recovery_8

TEACHER_RUN            = 'index_cued_first_diffusion_0.3_swap_7'
TEACHER_DIRECTION_IDX  = 0     # ablate direction 0 (matches MODEL_A)

STUDENT_RUN            = 'index_cued_first_diffusion_0.3_swap_recovery_8'

N_SAMPLES   = 512        # must match cached trajectories
N_TRIALS    = 288        # must match cached trajectories
DEVICE      = 'cuda'

# ── Which trial to inspect in the per-trial plots ─────────────────────────────
# Trial 6: cue=1, color1=0°, color2=180° (maximally distinct items)
TRIAL_IDX   = 6

# ── How many sample trajectories to DRAW in cloud/PCA plots ───────────────────
N_SAMPLES_PLOT = 30
N_PAIRS_SHOW   = 30
N_HIGHLIGHT  = 5

# ── Paths ─────────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
REPO_ROOT  = Path('/scratch3/shaiq_home/repos/behaviour_ddpm')
CACHE_DIR  = REPO_ROOT / 'ddpm/analysis/new_analysis/results/procrustes_heatmap/traj_cache'
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA

from ddpm.analysis.new_analysis.procrustes_heatmap import (
    align_trajectories, project_to_nullspace
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load trajectories and run alignment

In [ ]:
def _cache_path(label, direction_idx, n_samples):
    suffix = f'abl{direction_idx:02d}' if direction_idx is not None else 'unablated'
    return CACHE_DIR / f'{label}_{suffix}_S{n_samples}.npy'

# Load teacher trajectories (16-D)
t_path = _cache_path(TEACHER_RUN, TEACHER_DIRECTION_IDX, N_SAMPLES)
s_path = _cache_path(STUDENT_RUN, None, N_SAMPLES)
print(f'Teacher: {t_path.name}  exists={t_path.exists()}')
print(f'Student: {s_path.name}  exists={s_path.exists()}')

T16 = np.load(t_path)   # (N, S, T, 16)
S16 = np.load(s_path)   # (N, S, T, 16)
print(f'Loaded. Teacher shape: {T16.shape}   Student shape: {S16.shape}')

In [ ]:
# Load teacher nullspace and project to 14-D
import json
results_root = REPO_ROOT / 'results_link_sampler'
with open(results_root / TEACHER_RUN / 'nullspace_and_projection.json') as f:
    ns_data = json.load(f)
vecs = ns_data['nullspace_vectors']['vectors']
nullspace = np.stack([np.array(vecs[k]) for k in sorted(vecs)]).astype(np.float64)  # (14,16)
print(f'Nullspace shape: {nullspace.shape}')

X = project_to_nullspace(T16, nullspace).astype(np.float32)   # (N, S, T, 14)
Y = project_to_nullspace(S16, nullspace).astype(np.float32)   # (N, S, T, 14)
print(f'Projected: X={X.shape}  Y={Y.shape}')

In [ ]:
# Run alignment (GPU; ~5-10 s from projected arrays)
result = align_trajectories(
    X, Y,
    allow_scaling=True,
    n_restarts=3,
    max_iter=50,
    tol=1e-5,
    n_jobs=-1,
    seed=42,
    device=DEVICE,
)

print(f'Residual (aligned):  {result.residual:.4f}')
print(f'Residual (identity): {result.identity_residual:.4f}')
print(f'Scale c:             {result.c:.4f}')
print(f'Objective trace:     {[f"{v:.4f}" for v in result.objective_trace]}')
print(f'Restart residuals:   {[f"{v:.4f}" for v in result.restart_residuals]}')

In [ ]:
# Unpack alignment results
R       = result.R          # (14, 14) orthogonal rotation
c       = result.c          # positive scale
matches = result.matches    # (N, S) int — matches[n, k] = student sample for teacher k

N, S, T, M = X.shape

# Apply the same per-trajectory centering that align_trajectories did internally.
# Each trajectory is shifted so its temporal mean is zero — this is what makes the
# rotation well-defined (Procrustes cannot absorb translations, only rotations).
X_c = X - X.mean(axis=2, keepdims=True)   # (N, S, T, 14)
Y_c = Y - Y.mean(axis=2, keepdims=True)

# Build arrays for the selected trial in teacher's natural (centred) frame.
n = TRIAL_IDX
X_n    = X_c[n]                              # (S, T, 14) centred teacher samples
Y_n    = Y_c[n]                              # (S, T, 14) centred student samples
Y_match_n      = Y_n[matches[n]]             # (S, T, 14) student reordered by Hungarian
Y_aligned_n    = (Y_match_n @ R.T) / c       # (S, T, 14) matched student → teacher frame
Y_natural_aligned = (Y_n @ R.T) / c          # (S, T, 14) all student samples → teacher frame

# Xr_n kept only for the cost-matrix cell (uses original Procrustes cost definition)
Xr_n = c * (X_n @ R)

print(f'Trial {n}: X_n={X_n.shape}  Y_aligned_n={Y_aligned_n.shape}')
print(f'Teacher centred RMS: {np.sqrt((X_n**2).mean()):.4f}')
print(f'Student centred RMS: {np.sqrt((Y_n**2).mean()):.4f}')
print(f'Scale c={c:.4f}  →  teacher is {1/c:.2f}x student amplitude')

## 2. Convergence trace

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(result.objective_trace, '-o', color='steelblue')
ax.axhline(result.identity_residual, color='grey', ls='--', label=f'Identity baseline ({result.identity_residual:.3f})')
ax.set_xlabel('Alternating-min iteration')
ax.set_ylabel('Normalised residual')
ax.set_title('Convergence trace (best restart)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2b. Per-trial convergence (trial `TRIAL_IDX` only)

The §2 trace is the **global** objective across all 288 trials — bimodal trials are
diluted by the many unimodal ones.  Here we re-run alignment on just trial `TRIAL_IDX`
to see whether Hungarian assignment produces a larger improvement when the sample cloud
genuinely splits into two modes.

In [ ]:
# Re-run alignment on just trial TRIAL_IDX (N=1) to see the per-trial convergence.
# The global result above was optimised over all 288 trials simultaneously, which
# dilutes the signal from bimodal trials. Here we isolate trial n.

result_n = align_trajectories(
    X[[n]], Y[[n]],          # (1, S, T, 14) — single trial
    allow_scaling=True,
    n_restarts=3,
    max_iter=50,
    tol=1e-5,
    n_jobs=-1,
    seed=42,
    device=DEVICE,
)
print(f'Trial {n} (cue=1, 0°/180°):')
print(f'  Per-trial residual:  {result_n.residual:.4f}   c={result_n.c:.4f}')
print(f'  Global  residual:    {result.residual:.4f}   c={result.c:.4f}')
print(f'  Per-trial convergence: {[f"{v:.4f}" for v in result_n.objective_trace]}')
print(f'  Global  convergence:   {[f"{v:.4f}" for v in result.objective_trace]}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(result_n.objective_trace, '-o', color='steelblue', lw=2,
        label=f'Trial {n} only  (residual={result_n.residual:.3f})')
ax.plot(result.objective_trace,   '--o', color='grey', alpha=0.8, lw=2,
        label=f'Global all 288  (residual={result.residual:.3f})')
ax.axhline(result_n.identity_residual, color='steelblue', ls=':', alpha=0.6,
           label=f'Identity baseline (trial {n})')
ax.set_xlabel('Alternating-min iteration')
ax.set_ylabel('Normalised residual')
ax.set_title(f'Per-trial vs global convergence  (trial {n}: cue=1, 0°/180°)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. PCA setup for the selected trial

Fit a joint spatial PCA on teacher (natural frame) + student (rotated into teacher frame).
Both sets are now in the teacher's coordinate system so axes are directly interpretable.

In [ ]:
# Fit spatial PCA on teacher (natural) + student (rotated to teacher frame).
# Both are in the teacher's 14-D nullspace coordinate system.
spatial_data = np.concatenate([
    X_n.reshape(-1, M),
    Y_aligned_n.reshape(-1, M),
], axis=0)   # (2*S*T, 14)
spca = PCA(n_components=3)
spca.fit(spatial_data)
print(f'Spatial PCA explained variance (3 PCs): {spca.explained_variance_ratio_.cumsum()[-1]*100:.1f}%')
print(f'Per-PC explained: {[f"{v*100:.1f}%" for v in spca.explained_variance_ratio_]}')

def project_spatial(traj):
    """(S, T, M) → (S, T, 3): spatial PCA applied at each timestep."""
    s = traj.shape[0]
    flat = traj.reshape(s * T, M)
    return spca.transform(flat).reshape(s, T, 3)

## 4. Before vs after Hungarian permutation

Both teacher and student are shown in the **teacher's natural nullspace frame** (student rotated via R, rescaled by 1/c).

**Left**: student samples in natural index order (no correspondence assumed).  
**Right**: student samples reordered by Hungarian optimal assignment.

Lines connect the endpoint of each teacher trajectory to its paired student endpoint.

In [ ]:
from ddpm.analysis.new_analysis.procrustes_heatmap import _compute_cost_matrices_gpu
import torch

SHOW = 64

# Use centred data — same as what align_trajectories computed the cost on
X_t = torch.as_tensor(X_c[[n]], dtype=torch.float32, device=DEVICE)
Y_t = torch.as_tensor(Y_c[[n]], dtype=torch.float32, device=DEVICE)
R_t = torch.as_tensor(R, dtype=torch.float32, device=DEVICE)
cost_full = _compute_cost_matrices_gpu(X_t, Y_t, R_t, c)[0].cpu().numpy()  # (S, S)

cost_sub   = cost_full[:SHOW, :SHOW]
assign_sub = matches[n, :SHOW]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.imshow(cost_sub, aspect='auto', cmap='viridis', origin='upper')
plt.colorbar(im, ax=ax, label='Squared Frobenius cost')
for k in range(SHOW):
    j = assign_sub[k]
    if j < SHOW:
        ax.plot(j, k, 'r+', ms=4, mew=1)
ax.set_xlabel(f'Student sample index (0–{SHOW-1})')
ax.set_ylabel(f'Teacher sample index (0–{SHOW-1})')
ax.set_title(f'Cost matrix [{SHOW}×{SHOW} submatrix]\n(red + = Hungarian assignment)')

ax2 = axes[1]
assigned_costs = cost_full[np.arange(S), matches[n]]
random_costs   = cost_full[np.arange(S), rng.permutation(S)]
ax2.hist(assigned_costs, bins=60, alpha=0.6, color='steelblue',
         label=f'Hungarian  (mean={assigned_costs.mean():.2f})', density=True)
ax2.hist(random_costs,   bins=60, alpha=0.5, color='tomato',
         label=f'Random     (mean={random_costs.mean():.2f})',   density=True)
ax2.set_xlabel('Per-pair squared Frobenius cost')
ax2.set_ylabel('Density')
ax2.set_title('Assigned vs random pairing cost distribution')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

fig.suptitle(f'Trial {n} — assignment analysis (centred trajectories)', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Cost matrix and Hungarian assignment (trial `TRIAL_IDX`)

The full 512×512 cost matrix is too dense to read, so we show a 64×64 submatrix
(first 64 teacher vs first 64 student samples) with the assignment overlaid.

In [ ]:
from ddpm.analysis.new_analysis.procrustes_heatmap import _compute_cost_matrices_gpu
import torch

SHOW = 64   # submatrix size

X_t = torch.as_tensor(X[[n]], dtype=torch.float32, device=DEVICE)   # (1, S, T, 14)
Y_t = torch.as_tensor(Y[[n]], dtype=torch.float32, device=DEVICE)
R_t = torch.as_tensor(R, dtype=torch.float32, device=DEVICE)
cost_full = _compute_cost_matrices_gpu(X_t, Y_t, R_t, c)[0].cpu().numpy()  # (S, S)

cost_sub = cost_full[:SHOW, :SHOW]
assign_sub = matches[n, :SHOW]   # student index matched to each of first SHOW teachers

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: cost submatrix
ax = axes[0]
im = ax.imshow(cost_sub, aspect='auto', cmap='viridis', origin='upper')
plt.colorbar(im, ax=ax, label='Squared Frobenius cost')
# Overlay assignment for those teacher rows whose matched student is within SHOW
for k in range(SHOW):
    j = assign_sub[k]
    if j < SHOW:
        ax.plot(j, k, 'r+', ms=4, mew=1)
ax.set_xlabel(f'Student sample index (0–{SHOW-1})')
ax.set_ylabel(f'Teacher sample index (0–{SHOW-1})')
ax.set_title(f'Cost matrix [{SHOW}×{SHOW} submatrix]\n(red + = Hungarian assignment)')

# Right: distribution of per-pair costs (assigned vs random)
ax2 = axes[1]
assigned_costs = cost_full[np.arange(S), matches[n]]   # (S,) — cost of each assigned pair
random_costs   = cost_full[np.arange(S), rng.permutation(S)]  # (S,) — random pairing cost
ax2.hist(assigned_costs, bins=60, alpha=0.6, color='steelblue', label=f'Hungarian  (mean={assigned_costs.mean():.2f})', density=True)
ax2.hist(random_costs,   bins=60, alpha=0.5, color='tomato',    label=f'Random     (mean={random_costs.mean():.2f})',   density=True)
ax2.set_xlabel('Per-pair squared Frobenius cost')
ax2.set_ylabel('Density')
ax2.set_title('Assigned vs random pairing cost distribution')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

fig.suptitle(f'Trial {n} — assignment analysis', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Per-timestep residual distribution across all matched pairs

For every matched pair `(k, matches[n,k])`, compute `||X[n,k] − Y_aligned[n,k]||` at each timestep.
Both are in the teacher's frame, so residuals are directly interpretable in teacher-amplitude units.
Compare to identity pairing and to random pairing (both student sets also rotated to teacher frame).

In [ ]:
# Per-pair, per-timestep L2 residual in teacher frame  (S, T)
rand_perm   = rng.permutation(S)
diff_match  = X_n - Y_aligned_n                            # Hungarian matched
diff_id     = X_n - Y_natural_aligned                      # identity order
diff_rand   = X_n - Y_natural_aligned[rand_perm]           # random order

resid_match = np.linalg.norm(diff_match, axis=-1)          # (S, T)
resid_id    = np.linalg.norm(diff_id,    axis=-1)
resid_rand  = np.linalg.norm(diff_rand,  axis=-1)

t_axis = np.arange(T)

fig, ax = plt.subplots(figsize=(10, 4))
for resid, color, label in [
    (resid_match, 'steelblue', 'Hungarian matching'),
    (resid_id,    'goldenrod', 'Identity matching (k→k)'),
    (resid_rand,  'tomato',    'Random matching'),
]:
    med = np.median(resid, axis=0)
    p25 = np.percentile(resid, 25, axis=0)
    p75 = np.percentile(resid, 75, axis=0)
    ax.plot(t_axis, med, color=color, label=f'{label} (median)', lw=2)
    ax.fill_between(t_axis, p25, p75, color=color, alpha=0.15)

ax.set_xlabel('Prep timestep')
ax.set_ylabel('L2 residual (teacher-frame units)')
ax.set_title(f'Trial {n} — per-timestep residual  (median ± IQR across {S} pairs)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Best and worst matched pairs

Show the individual teacher + matched-student trajectories for the `N_HIGHLIGHT`
pairs with the **lowest** (best) and **highest** (worst) total matching cost.

In [ ]:
pair_costs  = cost_full[np.arange(S), matches[n]]   # (S,)  original Procrustes cost
best_idx    = np.argsort(pair_costs)[:N_HIGHLIGHT]
worst_idx   = np.argsort(pair_costs)[-N_HIGHLIGHT:][::-1]

def plot_pairs(ax, pair_indices, cmap_teacher='Blues', cmap_student='Oranges'):
    colors_t = plt.get_cmap(cmap_teacher)(np.linspace(0.4, 0.9, len(pair_indices)))
    colors_s = plt.get_cmap(cmap_student)(np.linspace(0.4, 0.9, len(pair_indices)))
    for i, k in enumerate(pair_indices):
        t_pc = spca.transform(X_n[k])           # (T, 3) teacher natural
        s_pc = spca.transform(Y_aligned_n[k])   # (T, 3) matched student in teacher frame
        cost = pair_costs[k]
        ax.plot(t_pc[:, 0], t_pc[:, 1], '-o', ms=2, color=colors_t[i],
                label=f'T[{k}] cost={cost:.2f}')
        ax.plot(s_pc[:, 0], s_pc[:, 1], '--s', ms=2, color=colors_s[i])
        ax.annotate('', xy=(s_pc[0,0], s_pc[0,1]), xytext=(t_pc[0,0], t_pc[0,1]),
                    arrowprops=dict(arrowstyle='->', color='grey', lw=0.8))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, indices, title in [
    (axes[0], best_idx,  f'{N_HIGHLIGHT} best-matched pairs (lowest cost)'),
    (axes[1], worst_idx, f'{N_HIGHLIGHT} worst-matched pairs (highest cost)'),
]:
    plot_pairs(ax, indices)
    ax.set_xlabel('sPC1'); ax.set_ylabel('sPC2')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=6, loc='best')
    ax.grid(alpha=0.2)

fig.suptitle(f'Trial {n} — solid=teacher, dashed=matched student (teacher frame)  |  arrow = start', fontsize=10)
plt.tight_layout()
plt.show()

## 8. 3-D trajectory cloud (`N_SAMPLES_PLOT` matched pairs)

Teacher samples (blue) and matched student samples rotated into the teacher's frame (orange).
Both clouds are in the same coordinate system — overlap means good alignment.
Mean trajectory over all 512 samples shown in black.

# Aligned arrays for all trials using centred data
Y_matched_all = Y_c[np.arange(N)[:, None], matches]    # (N, S, T, 14)
Y_aligned_all = (Y_matched_all @ R.T) / c               # (N, S, T, 14) student → teacher frame

diff_all  = X_c - Y_aligned_all                         # (N, S, T, 14)
resid_all = np.linalg.norm(diff_all, axis=-1)           # (N, S, T)

resid_pooled = resid_all.reshape(N * S, T)

fig, ax = plt.subplots(figsize=(10, 4))
med = np.median(resid_pooled, axis=0)
p25 = np.percentile(resid_pooled, 25, axis=0)
p75 = np.percentile(resid_pooled, 75, axis=0)
p10 = np.percentile(resid_pooled, 10, axis=0)
p90 = np.percentile(resid_pooled, 90, axis=0)
t_axis = np.arange(T)
ax.fill_between(t_axis, p10, p90, color='steelblue', alpha=0.10, label='10–90th pctile')
ax.fill_between(t_axis, p25, p75, color='steelblue', alpha=0.25, label='IQR')
ax.plot(t_axis, med, color='steelblue', lw=2, label='Median')
ax.set_xlabel('Prep timestep')
ax.set_ylabel('L2 residual (teacher-frame, centred)')
ax.set_title(f'All {N} trials × {S} pairs — per-timestep residual after centering + Hungarian + Procrustes')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Mean residual (centred, teacher frame): {resid_pooled.mean():.4f}')

In [ ]:
fig = plt.figure(figsize=(12, 5))

for col, (traj, full_traj, label, color) in enumerate([
    (X_n[plot_idx],         X_n,          'Teacher (natural frame)',                 'steelblue'),
    (Y_aligned_n[plot_idx], Y_aligned_n,  'Student (rotated to teacher frame)',      'tomato'),
]):
    ax = fig.add_subplot(1, 2, col + 1, projection='3d')
    traj_pc = project_spatial(traj)
    for k in range(N_SAMPLES_PLOT):
        ax.plot(traj_pc[k, :, 0], traj_pc[k, :, 1], traj_pc[k, :, 2],
                color=color, alpha=0.3, lw=0.8)
    full_pc  = project_spatial(full_traj)
    mean_pc  = full_pc.mean(axis=0)
    ax.plot(mean_pc[:, 0], mean_pc[:, 1], mean_pc[:, 2],
            color='black', lw=2.5, label=f'Mean (all {S})', zorder=5)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('sPC1'); ax.set_ylabel('sPC2'); ax.set_zlabel('sPC3')
    ax.legend(fontsize=8)

fig.suptitle(f'Trial {n} — 3-D cloud ({N_SAMPLES_PLOT} of {S} shown, mean uses all)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Mean trajectories averaged over all S=512 samples for trial TRIAL_IDX.
# Mirrors compare_two_models_prep_trajectories.ipynb style.

X_mean_n = X_n.mean(axis=0)           # (T, 14) teacher mean
Y_mean_n = Y_aligned_n.mean(axis=0)   # (T, 14) student mean, teacher frame

X_mean_pc = spca.transform(X_mean_n)  # (T, 3)
Y_mean_pc = spca.transform(Y_mean_n)  # (T, 3)

fig = plt.figure(figsize=(14, 5))

# 2D: PC1 / PC2
ax1 = fig.add_subplot(1, 2, 1)
for k in range(N_SAMPLES_PLOT):
    x_s = project_spatial(X_n[[k]])[0, :, :2]
    y_s = project_spatial(Y_aligned_n[[k]])[0, :, :2]
    ax1.plot(x_s[:, 0], x_s[:, 1], color='steelblue', alpha=0.12, lw=0.6)
    ax1.plot(y_s[:, 0], y_s[:, 1], color='tomato',    alpha=0.12, lw=0.6)
ax1.plot(X_mean_pc[:, 0], X_mean_pc[:, 1], '-o', ms=4, color='steelblue', lw=2.5, label='Teacher mean')
ax1.plot(Y_mean_pc[:, 0], Y_mean_pc[:, 1], '-o', ms=4, color='tomato',    lw=2.5, label='Student mean')
ax1.plot(X_mean_pc[0, 0], X_mean_pc[0, 1], 'k^', ms=9, zorder=6, label='Start')
ax1.plot(Y_mean_pc[0, 0], Y_mean_pc[0, 1], 'k^', ms=9, zorder=6)
ax1.set_xlabel('sPC1'); ax1.set_ylabel('sPC2')
ax1.set_title(f'PC1/PC2  (mean ± {N_SAMPLES_PLOT} sample traces)')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# 3D
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
for k in range(N_SAMPLES_PLOT):
    x_s = project_spatial(X_n[[k]])[0]
    y_s = project_spatial(Y_aligned_n[[k]])[0]
    ax2.plot(x_s[:, 0], x_s[:, 1], x_s[:, 2], color='steelblue', alpha=0.12, lw=0.6)
    ax2.plot(y_s[:, 0], y_s[:, 1], y_s[:, 2], color='tomato',    alpha=0.12, lw=0.6)
ax2.plot(X_mean_pc[:, 0], X_mean_pc[:, 1], X_mean_pc[:, 2],
         '-o', ms=4, color='steelblue', lw=2.5, label='Teacher mean')
ax2.plot(Y_mean_pc[:, 0], Y_mean_pc[:, 1], Y_mean_pc[:, 2],
         '-o', ms=4, color='tomato',    lw=2.5, label='Student mean')
ax2.set_xlabel('sPC1'); ax2.set_ylabel('sPC2'); ax2.set_zlabel('sPC3')
ax2.set_title('3D')
ax2.legend(fontsize=8)

fig.suptitle(
    f'Trial {n}  (cue=1, 0°/180°) — aligned mean trajectory  '
    f'[Procrustes residual={result.residual:.3f},  c={c:.3f}]',
    fontsize=11,
)
plt.tight_layout()
plt.show()

print(f'Mean-trajectory RMSE (nullspace units): {np.sqrt(((X_mean_n - Y_mean_n)**2).mean()):.4f}')

## 9. Residual aggregated across all trials

Per-timestep residual (median ± IQR) averaged over **all N trials**,
not just `TRIAL_IDX` — gives a population-level picture.

In [ ]:
# Aligned arrays for all trials — student rotated into teacher frame
Y_matched_all  = Y[np.arange(N)[:, None], matches]     # (N, S, T, 14)
Y_aligned_all  = (Y_matched_all @ R.T) / c              # (N, S, T, 14) student in teacher frame

diff_all  = X - Y_aligned_all                           # (N, S, T, 14)
resid_all = np.linalg.norm(diff_all, axis=-1)           # (N, S, T)

resid_pooled = resid_all.reshape(N * S, T)

fig, ax = plt.subplots(figsize=(10, 4))
med = np.median(resid_pooled, axis=0)
p25 = np.percentile(resid_pooled, 25, axis=0)
p75 = np.percentile(resid_pooled, 75, axis=0)
p10 = np.percentile(resid_pooled, 10, axis=0)
p90 = np.percentile(resid_pooled, 90, axis=0)
t_axis = np.arange(T)
ax.fill_between(t_axis, p10, p90, color='steelblue', alpha=0.10, label='10–90th pctile')
ax.fill_between(t_axis, p25, p75, color='steelblue', alpha=0.25, label='IQR')
ax.plot(t_axis, med, color='steelblue', lw=2, label='Median')
ax.set_xlabel('Prep timestep')
ax.set_ylabel('L2 residual (teacher-frame units)')
ax.set_title(f'All {N} trials × {S} pairs — per-timestep residual after Hungarian+Procrustes alignment')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Mean residual (teacher frame): {resid_pooled.mean():.4f}')